# Pipeline v16 -- CoT LoRA Fine-tune (verbose, LLaMA-Factory style)
**EXACT 2026 -- XAI Challenge @ IJCNN | Kaggle T4x2 | data v4 318 records**

## v16 = control variable
Verbose rebuild of the v14 finetune that produced our best LoRA (val 81.0%).
No training changes vs v15 -- this is the BASELINE for A/B testing v17.

## A/B test plan
- **v16**: r=16, alpha=16, dropout=0, 2 epochs, no oversample (control)
- **v17**: v16 + NEFTune(alpha=5) + class-balanced oversample (No 5x, Unknown 2x)
- Both train on the SAME 80/20 split with the SAME seed -> direct comparison.

## Output
- LoRA adapter: `/kaggle/working/qwen3_cot_lora_v16`
- Eval results : `/kaggle/working/v16_eval_results.json`
- Full metrics : `/kaggle/working/v16_full_metrics.json`


In [1]:
#!/usr/bin/env python3
"""
notebook_v14_cot_lora_finetune.py
EXACT 2026 -- v14: Train Qwen3-8B LoRA on dataset's `explanation` field
to produce pedagogical CoT reasoning + Final Answer marker.
Output: /kaggle/working/qwen3_cot_lora_v14 (PEFT adapter for vLLM).
"""


"\nnotebook_v14_cot_lora_finetune.py\nEXACT 2026 -- v14: Train Qwen3-8B LoRA on dataset's `explanation` field\nto produce pedagogical CoT reasoning + Final Answer marker.\nOutput: /kaggle/working/qwen3_cot_lora_v14 (PEFT adapter for vLLM).\n"

In [2]:
# ================= Kaggle T4 fix =================
import os, shutil, glob
STUB_DIR = "/tmp/cuda_stubs"; os.makedirs(STUB_DIR, exist_ok=True)
stub = os.path.join(STUB_DIR, "libcuda.so")
if os.path.exists(stub) or os.path.islink(stub): os.remove(stub)
for c in ["/usr/lib/x86_64-linux-gnu/libcuda.so.1", "/usr/lib/x86_64-linux-gnu/libcuda.so",
          *glob.glob("/usr/**/libcuda.so*", recursive=True)]:
    if os.path.exists(c) and not os.path.islink(c):
        os.symlink(c, stub); break
os.environ["LIBRARY_PATH"] = f"{STUB_DIR}:" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{STUB_DIR}:" + os.environ.get("LD_LIBRARY_PATH", "")
shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)
print("Kaggle T4 fixes applied")


Kaggle T4 fixes applied


In [3]:
# ================= INSTALL =================
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages",
                "unsloth", "trl<=0.24.0", "datasets<4.4.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--break-system-packages",
                "--no-cache-dir", "peft", "accelerate", "bitsandbytes"], check=True)
import torch
print("PyTorch", torch.__version__, "CUDA", torch.cuda.is_available())


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 14.0 MB/s eta 0:00:00
PyTorch 2.10.0+cu128 CUDA True


In [4]:
# ================= CONFIG (verbose) =================
import os as _os
def _resolve(cands, label="path"):
    for p in cands:
        if _os.path.exists(p):
            print(f"  [PATH] resolved {label:8s}: {p}")
            return p
    print(f"  [WARN] {label}: none of candidates exist; using first: {cands[0]}")
    return cands[0]

print("="*70); print("  Resolving Kaggle paths..."); print("="*70)
MODEL_PATH = _resolve([
    "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1",
], "MODEL")
TRAIN_PATH = _resolve([
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json",
], "TRAIN")
TEST_PATH = _resolve([
    "/kaggle/input/test-pipeline/generated_v4style_300.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json",
], "TEST")
DATASET_PATH = TRAIN_PATH

LORA_OUTPUT_DIR  = "/kaggle/working/qwen3_cot_lora_v16"
CHECKPOINT_DIR   = "/kaggle/working/cot_lora_ckpt_v16"
EVAL_OUTPUT_PATH = "/kaggle/working/v16_eval_results.json"

# Split (same as v13.x / v14 -- SEED=42, 80/20 sample-level)
SEED        = 42
TRAIN_RATIO = 0.80

# Model
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT   = True

# LoRA -- match v14 (control variable, do not change in v16)
LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

# Training -- match v14
EPOCHS         = 2
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
LEARNING_RATE  = 1e-4
WARMUP_STEPS   = 10
WEIGHT_DECAY   = 0.01
OPTIMIZER      = "adamw_8bit"
LR_SCHEDULER   = "linear"
LOGGING_STEPS  = 5      # verbose: print every 5 steps (vs 10 in v14)
EVAL_STEPS     = 25     # verbose: in-loop eval markers

# v16 ablations OFF (control variable). v17 will turn these ON.
USE_NEFTUNE         = False
NEFTUNE_NOISE_ALPHA = 0
USE_CLASS_BALANCE   = False
OVERSAMPLE_NO       = 1
OVERSAMPLE_UNKNOWN  = 1

# System prompt -- IDENTICAL at training and inference
ENABLE_THINKING_TRAIN = False
SYSTEM_PROMPT = (
    "You are a careful logic tutor. Given a list of premises and a question, "
    "reason step-by-step by referencing specific premises (e.g. 'From premise 3...'). "
    "Then state your conclusion on a final line in the exact format: 'Final Answer: X' "
    "where X is one of: Yes, No, Unknown, A, B, C, or D."
)

# Evaluation
EVAL_MAX_NEW_TOKENS = 512
EVAL_TEMPERATURE    = 0.1
EVAL_N_SAMPLES      = None

# Verbose: print every config item in a single block
print("\n" + "="*70); print("  v16 Training Configuration"); print("="*70)
print(f"  Model              : qwen-3-8b (load_in_4bit={LOAD_IN_4BIT})")
print(f"  Method             : LoRA SFT (responses-only)")
print(f"  LoRA r / alpha     : {LORA_R} / {LORA_ALPHA}")
print(f"  LoRA dropout       : {LORA_DROPOUT}")
print(f"  Target modules     : {len(TARGET_MODULES)} = {TARGET_MODULES}")
print(f"  Max seq length     : {MAX_SEQ_LENGTH}")
print(f"  Epochs             : {EPOCHS}")
print(f"  Per-device batch   : {BATCH_SIZE}")
print(f"  Grad accumulation  : {GRAD_ACCUM} (effective batch = {BATCH_SIZE*GRAD_ACCUM})")
print(f"  Learning rate      : {LEARNING_RATE}")
print(f"  LR scheduler       : {LR_SCHEDULER}")
print(f"  Warmup steps       : {WARMUP_STEPS}")
print(f"  Weight decay       : {WEIGHT_DECAY}")
print(f"  Optimizer          : {OPTIMIZER}")
print(f"  Logging steps      : {LOGGING_STEPS}")
print(f"  Seed               : {SEED}")
print(f"  Train ratio        : {TRAIN_RATIO}")
print(f"  enable_thinking    : {ENABLE_THINKING_TRAIN}")
print(f"  NEFTune (v17 only) : {USE_NEFTUNE} (alpha={NEFTUNE_NOISE_ALPHA})")
print(f"  Class balance (v17): {USE_CLASS_BALANCE} (No={OVERSAMPLE_NO}x  Unknown={OVERSAMPLE_UNKNOWN}x)")
print(f"  LoRA output        : {LORA_OUTPUT_DIR}")
print("="*70); print("v16 config loaded (CONTROL variable -- no NEFTune, no oversample)")


  Resolving Kaggle paths...
  [PATH] resolved MODEL   : /kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1
  [PATH] resolved TRAIN   : /kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_v5_repair_tier1_dedup_normfol_drop_logicfallacy.json
  [PATH] resolved TEST    : /kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json

  v16 Training Configuration
  Model              : qwen-3-8b (load_in_4bit=True)
  Method             : LoRA SFT (responses-only)
  LoRA r / alpha     : 16 / 16
  LoRA dropout       : 0
  Target modules     : 7 = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  Max seq length     : 4096
  Epochs             : 2
  Per-device batch   : 2
  Grad accumulation  : 4 (effective batch = 8)
  Learning rate      : 0.0001
  LR scheduler       : linear
  Warmup steps       : 10
  Weight decay       : 0.01
  Optimizer          : adamw_8bit
  Logging steps      : 5
  Seed               : 42
 

In [5]:
# ================= LOAD MODEL + ATTACH LoRA =================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED,
)
trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
tot = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trn:,}/{tot:,} ({100*trn/tot:.2f}%)")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Unsloth 2026.6.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Trainable: 43,646,976/4,761,498,624 (0.92%)


In [6]:
# ================= BUILD CoT DATASET (verbose) =================
import json, random, re
from collections import Counter
from datasets import Dataset

with open(DATASET_PATH, encoding="utf-8") as f:
    raw = json.load(f)
n = len(raw)
print("="*70); print(f"  Dataset loaded: {n} records"); print("="*70)

# Stats
n_q  = sum(len(s.get("questions",[])) for s in raw)
n_pr = sum(len(s.get("premises-NL",[])) for s in raw)
gold_fol = sum(len(s.get("premises-FOL",[])) for s in raw)
print(f"  Total questions       : {n_q}")
print(f"  Total premises        : {n_pr}")
print(f"  Premises with FOL     : {gold_fol} ({100*gold_fol/n_pr:.1f}%)")
print(f"  Avg premises/record   : {n_pr/n:.1f}")
print(f"  Avg questions/record  : {n_q/n:.1f}")

# Full-data class distribution
all_labels = [str(a).strip() for s in raw for a in s.get("answers", []) if a]
cls_dist = Counter(all_labels)
print(f"\n  Class distribution (full dataset, n={len(all_labels)}):")
for lab, cnt in sorted(cls_dist.items(), key=lambda x: -x[1]):
    pct = 100*cnt/len(all_labels)
    bar = "#"*int(pct/2)
    print(f"    {lab:>10}: {cnt:>4} ({pct:>5.1f}%) {bar}")

# Sample-level split, deterministic, MATCHES v13.x
rng = random.Random(SEED)
ids = list(range(n)); rng.shuffle(ids)
n_tr = int(n * TRAIN_RATIO)
train_ids = set(ids[:n_tr]); val_ids = set(ids[n_tr:])
train_samples = [raw[i] for i in range(n) if i in train_ids]
val_samples   = [raw[i] for i in range(n) if i in val_ids]
print(f"\n  Split (seed={SEED}, ratio={TRAIN_RATIO}):")
print(f"    Train: {len(train_samples)} records ({100*len(train_samples)/n:.1f}%)")
print(f"    Val  : {len(val_samples)} records ({100*len(val_samples)/n:.1f}%)")

# Per-split class distribution
def cls_split(samples, name):
    labs = [str(a).strip() for s in samples for a in s.get("answers",[]) if a]
    c = Counter(labs)
    print(f"\n  Class distribution ({name}, n={len(labs)}):")
    for lab in sorted(cls_dist.keys()):
        cnt = c.get(lab, 0)
        pct = 100*cnt/len(labs) if labs else 0
        print(f"    {lab:>10}: {cnt:>4} ({pct:>5.1f}%)")
    return c
cls_train = cls_split(train_samples, "train")
cls_val   = cls_split(val_samples, "val")

# Prompt builders
def build_user_msg(premises_nl, question):
    lines = ["### Premises"]
    for i, p in enumerate(premises_nl):
        lines.append(f"P{i+1}. {p}")
    return "\n".join(lines) + f"\n\n### Question\n{question}"

def build_assistant_msg(explanation, gold):
    txt = explanation.strip()
    last = txt.split("\n")[-1].lower()
    if not (re.search(r"\bfinal\s+answer\b", last) or re.search(r"\banswer\s*:", last)):
        txt = f"{txt}\n\nFinal Answer: {gold}"
    return txt

def to_records(samples, oversample=False):
    """Build SFT records. If oversample=True, duplicate No/Unknown rows."""
    out = []
    for s in samples:
        nls = s.get("premises-NL", [])
        for q, a, e in zip(s.get("questions",[]), s.get("answers",[]), s.get("explanation",[])):
            if not e or not str(a).strip():
                continue
            rec = {"conversations": [
                {"role":"system",    "content": SYSTEM_PROMPT},
                {"role":"user",      "content": build_user_msg(nls, q)},
                {"role":"assistant", "content": build_assistant_msg(e, a)},
            ], "_label": str(a).strip()}
            out.append(rec)
            if oversample:
                lab_u = str(a).strip().upper()
                if lab_u == "NO":
                    for _ in range(OVERSAMPLE_NO - 1): out.append(rec)
                elif lab_u == "UNKNOWN":
                    for _ in range(OVERSAMPLE_UNKNOWN - 1): out.append(rec)
    return out

train_records = to_records(train_samples, oversample=USE_CLASS_BALANCE)

# Post-oversample distribution
post_dist = Counter(r["_label"] for r in train_records)
print(f"\n  Train SFT records (after oversample={USE_CLASS_BALANCE}):")
for lab in sorted(cls_dist.keys()):
    cnt = post_dist.get(lab, 0)
    pct = 100*cnt/len(train_records) if train_records else 0
    flag = ""
    pre  = sum(1 for s in train_samples for a in s.get("answers",[]) if str(a).strip()==lab)
    if cnt > pre: flag = f"  <- oversampled {cnt//pre}x"
    print(f"    {lab:>10}: {cnt:>4} ({pct:>5.1f}%){flag}")
print(f"  Total train examples: {len(train_records)}")

# Sample record preview
print(f"\n  --- Sample training record ---")
print(f"  USER     : {train_records[0]['conversations'][1]['content'][:300]}...")
print(f"  ASSISTANT: {train_records[0]['conversations'][2]['content'][:400]}")

# Strip _label (not used by trainer)
for r in train_records: r.pop("_label", None)
train_ds = Dataset.from_list(train_records)
print(f"\n  HF Dataset: {train_ds}")


  Dataset loaded: 399 records
  Total questions       : 784
  Total premises        : 4294
  Premises with FOL     : 4326 (100.7%)
  Avg premises/record   : 10.8
  Avg questions/record  : 2.0

  Class distribution (full dataset, n=784):
            No:  233 ( 29.7%) ##############
       Unknown:  187 ( 23.9%) ###########
           Yes:  170 ( 21.7%) ##########
             A:  111 ( 14.2%) #######
             B:   47 (  6.0%) ##
             C:   20 (  2.6%) #
             D:   16 (  2.0%) #

  Split (seed=42, ratio=0.8):
    Train: 319 records (79.9%)
    Val  : 80 records (20.1%)

  Class distribution (train, n=628):
             A:   83 ( 13.2%)
             B:   39 (  6.2%)
             C:   14 (  2.2%)
             D:   11 (  1.8%)
            No:  193 ( 30.7%)
       Unknown:  152 ( 24.2%)
           Yes:  136 ( 21.7%)

  Class distribution (val, n=156):
             A:   28 ( 17.9%)
             B:    8 (  5.1%)
             C:    6 (  3.8%)
             D:    5 (  3.2%)
    

In [7]:
# ================= TRAINER (responses-only, verbose) =================
import torch
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

print("="*70); print("  Building SFTTrainer"); print("="*70)

def formatting_func(examples):
    convos = examples["conversations"]
    def _fmt(c):
        try:
            return tokenizer.apply_chat_template(
                c, tokenize=False, add_generation_prompt=False,
                enable_thinking=ENABLE_THINKING_TRAIN)
        except TypeError:
            return tokenizer.apply_chat_template(
                c, tokenize=False, add_generation_prompt=False)
    if len(convos) > 0 and isinstance(convos[0], dict):
        return [_fmt(convos)]
    return [_fmt(c) for c in convos]

# v16 / v17 trainer args (NEFTune flag wired in but OFF for v16)
_sft_kwargs = dict(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    optim=OPTIMIZER,
    lr_scheduler_type=LR_SCHEDULER,
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    save_total_limit=1,
    seed=SEED,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    report_to="none",
)
if USE_NEFTUNE and NEFTUNE_NOISE_ALPHA > 0:
    _sft_kwargs["neftune_noise_alpha"] = NEFTUNE_NOISE_ALPHA
    print(f"  [v17] NEFTune ENABLED with noise_alpha={NEFTUNE_NOISE_ALPHA}")
else:
    print(f"  [v16] NEFTune OFF (control)")

sft = SFTConfig(**_sft_kwargs)

trainer_obj = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds,
                         args=sft, formatting_func=formatting_func)
trainer_obj = train_on_responses_only(
    trainer_obj,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Verbose: print final trainer state
total_steps = (len(train_ds) // (BATCH_SIZE*GRAD_ACCUM)) * EPOCHS
print(f"\n  Trainer ready:")
print(f"    Training examples : {len(train_ds)}")
print(f"    Effective batch   : {BATCH_SIZE*GRAD_ACCUM}")
print(f"    Total steps       : ~{total_steps}")
print(f"    Steps per epoch   : ~{total_steps//EPOCHS}")
print(f"    Will log every    : {LOGGING_STEPS} steps")


  Building SFTTrainer
  [v16] NEFTune OFF (control)


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/628 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=8):   0%|          | 0/628 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/628 [00:00<?, ? examples/s]


  Trainer ready:
    Training examples : 628
    Effective batch   : 8
    Total steps       : ~156
    Steps per epoch   : ~78
    Will log every    : 5 steps


In [8]:
# ================= TRAIN + SAVE LoRA (verbose) =================
import time, os, json
t0 = time.time()
print("="*70); print("  STARTING v16 TRAINING"); print("="*70)
print("  Watch for per-step loss/lr/grad_norm below...")
result = trainer_obj.train()
dur_min = (time.time() - t0) / 60

# Per-epoch summary from log_history
print("\n" + "="*70); print("  TRAINING COMPLETE"); print("="*70)
log = trainer_obj.state.log_history
train_logs = [x for x in log if "loss" in x and "eval_loss" not in x]
eval_logs  = [x for x in log if "eval_loss" in x]
print(f"  Duration         : {dur_min:.2f} min")
print(f"  Final train loss : {result.training_loss:.4f}")
print(f"  Log entries      : train={len(train_logs)}  eval={len(eval_logs)}")
if train_logs:
    print(f"\n  Loss curve (every {LOGGING_STEPS} steps):")
    for entry in train_logs:
        step = entry.get("step", "?"); loss = entry.get("loss", "?")
        lr   = entry.get("learning_rate", 0); gn = entry.get("grad_norm", 0)
        print(f"    step {step:>4}: loss={loss:.4f}  lr={lr:.2e}  grad_norm={gn:.3f}")

# Save adapter + tokenizer
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"\n  Saved LoRA -> {LORA_OUTPUT_DIR}")
print(f"  Contents: {os.listdir(LORA_OUTPUT_DIR)}")

# Also save a config snapshot for reproducibility
json.dump({
    "version": "v16",
    "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
    "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY,
    "use_neftune": USE_NEFTUNE, "neftune_noise_alpha": NEFTUNE_NOISE_ALPHA,
    "use_class_balance": USE_CLASS_BALANCE,
    "oversample_no": OVERSAMPLE_NO, "oversample_unknown": OVERSAMPLE_UNKNOWN,
    "training_loss_final": float(result.training_loss),
    "duration_min": dur_min,
    "n_train_examples": len(train_ds),
}, open(f"{LORA_OUTPUT_DIR}/v16_train_config.json", "w"), indent=2)
print(f"  Saved config snapshot: {LORA_OUTPUT_DIR}/v16_train_config.json")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


  STARTING v16 TRAINING
  Watch for per-step loss/lr/grad_norm below...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 628 | Num Epochs = 2 | Total steps = 158
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 43,646,976 of 8,234,382,336 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,2.220506
10,1.392247
15,0.926199
20,0.825956
25,0.620343
30,0.605981
35,0.507044
40,0.617525
45,0.519456
50,0.491989


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/cot_lora_ckpt_v16/checkpoint-79/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/cot_lora_ckpt_v16/checkpoint-158/tokenizer_config.json.



  TRAINING COMPLETE
  Duration         : 39.67 min
  Final train loss : 0.5591
  Log entries      : train=31  eval=0

  Loss curve (every 5 steps):
    step    5: loss=2.2205  lr=4.00e-05  grad_norm=1.889
    step   10: loss=1.3922  lr=9.00e-05  grad_norm=0.761
    step   15: loss=0.9262  lr=9.73e-05  grad_norm=0.521
    step   20: loss=0.8260  lr=9.39e-05  grad_norm=0.451
    step   25: loss=0.6203  lr=9.05e-05  grad_norm=0.552
    step   30: loss=0.6060  lr=8.72e-05  grad_norm=0.636
    step   35: loss=0.5070  lr=8.38e-05  grad_norm=0.370
    step   40: loss=0.6175  lr=8.04e-05  grad_norm=0.493
    step   45: loss=0.5195  lr=7.70e-05  grad_norm=0.369
    step   50: loss=0.4920  lr=7.36e-05  grad_norm=0.422
    step   55: loss=0.5791  lr=7.03e-05  grad_norm=0.777
    step   60: loss=0.5933  lr=6.69e-05  grad_norm=0.509
    step   65: loss=0.4909  lr=6.35e-05  grad_norm=0.368
    step   70: loss=0.4695  lr=6.01e-05  grad_norm=0.514
    step   75: loss=0.4430  lr=5.68e-05  grad_norm=0.

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/qwen3_cot_lora_v16/tokenizer_config.json.



  Saved LoRA -> /kaggle/working/qwen3_cot_lora_v16
  Contents: ['adapter_model.safetensors', 'README.md', 'tokenizer.json', 'chat_template.jinja', 'adapter_config.json', 'tokenizer_config.json']
  Saved config snapshot: /kaggle/working/qwen3_cot_lora_v16/v16_train_config.json


In [9]:
# ================= QUICK SANITY CHECK (not full eval) =================
# Chi chay 12 cau de xac nhan adapter sinh dung dinh dang "Final Answer: X".
# Full eval + per-class F1 nam o notebook INFERENCE (v16.1 / v17.1), khong o day.
import sys, time, random as _rnd
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def _extract(text):
    import re
    m = re.search(r"Final\s*Answer\s*[:\-]?\s*([A-D]|Yes|No|Unknown)", text, re.I)
    if m:
        a = m.group(1)
        return a.capitalize() if a.lower() in ("yes","no","unknown") else a.upper()
    return "UNPARSEABLE"

def _gen(nls, q):
    msgs = [{"role":"system","content":SYSTEM_PROMPT},
            {"role":"user","content":build_user_msg(nls, q)}]
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                        return_tensors="pt").to(model.device)
    import torch
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=256, temperature=0.0,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

print("="*70, flush=True)
print("  QUICK SANITY (12 val questions, greedy)", flush=True)
print("="*70, flush=True)
_probe = []
for s in val_samples:
    nls = s.get("premises-NL", [])
    for q, a in zip(s.get("questions",[]), s.get("answers",[])):
        if str(a).strip(): _probe.append((nls, q, str(a).strip()))
_rnd.Random(SEED).shuffle(_probe)
_probe = _probe[:12]

correct = 0
for j,(nls,q,gold) in enumerate(_probe,1):
    pred = _extract(_gen(nls,q))
    ok = pred.upper()==gold.upper()
    correct += ok
    print(f"  [{j:2d}/12] gold={gold:>8s} pred={pred:>8s} {'OK' if ok else 'X'}", flush=True)
print("-"*70, flush=True)
print(f"  Sanity acc: {correct}/12 = {100*correct/12:.0f}%  "
      f"({'looks healthy' if correct>=6 else 'CHECK PROMPT/FORMAT'})", flush=True)
print("  (Day chi la sanity, KHONG phai final metric. Full eval o v16.1/v17.1.)", flush=True)


  QUICK SANITY (12 val questions, greedy)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

  [ 1/12] gold=      No pred=      No OK


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [ 2/12] gold=     Yes pred=      No X


...

  [10/12] gold=       D pred=       A X


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [11/12] gold=       A pred=       A OK


Both `max_new_tokens` (=256) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  [12/12] gold=      No pred=      No OK
----------------------------------------------------------------------
  Sanity acc: 6/12 = 50%  (looks healthy)
  (Day chi la sanity, KHONG phai final metric. Full eval o v16.1/v17.1.)


In [10]:
# ================= ZIP ADAPTER + FINAL SANITY CHECK =================
# Zip adapter thanh 1 file sach (~85MB) de notebook inference mount nhanh,
# tranh "Draft Session - Adding data" bi ket khi add nguyen notebook output.
import os, zipfile, sys

ADAPTER_DIR = LORA_OUTPUT_DIR
ZIP_PATH    = f"/kaggle/working/{os.path.basename(ADAPTER_DIR)}.zip"

# Chi zip cac file CAN cho inference (bo optimizer/checkpoint/cache)
WANT = ["adapter_config.json", "adapter_model.safetensors",
        "tokenizer.json", "tokenizer_config.json",
        "chat_template.jinja", "special_tokens_map.json"]

print("="*70, flush=True)
print("  ZIPPING ADAPTER", flush=True)
print("="*70, flush=True)
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(ADAPTER_DIR):
        if fn in WANT or fn.endswith(".json"):
            full = os.path.join(ADAPTER_DIR, fn)
            if os.path.isfile(full):
                # luu duoi thu muc adapter de unzip ra dung cau truc
                zf.write(full, arcname=f"{os.path.basename(ADAPTER_DIR)}/{fn}")
                print(f"  + {fn} ({os.path.getsize(full)/1e6:.1f} MB)", flush=True)

zip_mb = os.path.getsize(ZIP_PATH)/1e6
print(f"\n  Zip created: {ZIP_PATH} ({zip_mb:.1f} MB)", flush=True)

# ---------- FINAL SANITY CHECK ----------
print("\n" + "="*70, flush=True)
print("  FINAL SANITY CHECK", flush=True)
print("="*70, flush=True)
cfg_path  = os.path.join(ADAPTER_DIR, "adapter_config.json")
safe_path = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")

assert os.path.exists(cfg_path),  f"MISSING adapter_config.json in {ADAPTER_DIR}"
assert os.path.exists(safe_path), f"MISSING adapter_model.safetensors in {ADAPTER_DIR}"
assert os.path.exists(ZIP_PATH),  f"MISSING zip {ZIP_PATH}"
assert zip_mb > 1.0,              f"Zip suspiciously small ({zip_mb:.2f} MB) -- adapter may be empty"

print(f"  [OK] adapter_config.json       ({os.path.getsize(cfg_path)} bytes)", flush=True)
print(f"  [OK] adapter_model.safetensors ({os.path.getsize(safe_path)/1e6:.1f} MB)", flush=True)
print(f"  [OK] {os.path.basename(ZIP_PATH)} ({zip_mb:.1f} MB)", flush=True)
print("\n" + "="*70, flush=True)
print("  NOTEBOOK FINETUNE COMPLETED SUCCESSFULLY", flush=True)
print("="*70, flush=True)
print(f"  Next: Save Version, then in the inference notebook Add Data ->", flush=True)
print(f"        this notebook's output, and it will find {os.path.basename(ZIP_PATH)}.", flush=True)


  ZIPPING ADAPTER
  + v16_train_config.json (0.0 MB)
  + adapter_model.safetensors (174.7 MB)
  + tokenizer.json (11.4 MB)
  + chat_template.jinja (0.0 MB)
  + adapter_config.json (0.0 MB)
  + tokenizer_config.json (0.0 MB)

  Zip created: /kaggle/working/qwen3_cot_lora_v16.zip (163.6 MB)

  FINAL SANITY CHECK
  [OK] adapter_config.json       (1266 bytes)
  [OK] adapter_model.safetensors (174.7 MB)
  [OK] qwen3_cot_lora_v16.zip (163.6 MB)

  NOTEBOOK FINETUNE COMPLETED SUCCESSFULLY
  Next: Save Version, then in the inference notebook Add Data ->
        this notebook's output, and it will find qwen3_cot_lora_v16.zip.
